In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from matplotlib import pyplot as plt
from plotting import plot_learning_curve
from pointnet_model import pnet
from datetime import datetime
import click
import os

In [ ]:
from tensorflow import keras
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np

class OrthogonalRegularizer(keras.regularizers.Regularizer):
    def __init__(self, num_features, l2reg=0.001):
        self.num_features = num_features
        self.l2reg = l2reg
        self.eye = tf.eye(num_features)
    def __call__(self, x):
        x = tf.reshape(x, (-1, self.num_features, self.num_features))
        xxt = tf.tensordot(x, x, axes=(2, 2))
        xxt = tf.reshape(xxt, (-1, self.num_features, self.num_features))
        return tf.reduce_sum(self.l2reg * tf.square(xxt - self.eye))
    def get_config(self):
        return {'l2reg': self.l2reg, 'num_features': self.num_features}

##### Test Model Loading
Added this extra step to check whether full model loading is done correctling with the custom regularizer

In [ ]:
# Load the saved model using Keras API
saved_model_path = 'models/2024-04-04-16:29:16/full_model' 
model = keras.models.load_model(saved_model_path, custom_objects={'OrthogonalRegularizer': OrthogonalRegularizer})

# Check if the model has been loaded correctly
print(model.summary())

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
from datetime import datetime
import os
from sklearn.model_selection import train_test_split
from plotting import plot_learning_curve

# Load the saved model using Keras API
saved_model_path = 'models/2024-04-04-16:29:16/full_model' 
model = load_model(saved_model_path, custom_objects={'OrthogonalRegularizer': OrthogonalRegularizer})

num_classes = 3  # Number of classes for the new task
num_epochs = 10
batch_size = 32

# Reconstruct the model without the last layer and add a new output layer with a unique name
x = model.layers[-2].output
new_output = tf.keras.layers.Dense(num_classes, activation='softmax', name='reconstruction')(x)  # Enter a unique name here
model = tf.keras.models.Model(inputs=model.input, outputs=new_output)

# Load data and labels for the new task
train_data = np.load('O16_expt_downstream/ready/tpcnet_O16_expt_data_512.npy')
val_data = np.load('O16_expt_downstream/ready/tpcnet_O16_expt_labels_512.npy')

train_data, test_data, train_labels, test_labels = train_test_split(train_data, val_data, test_size=0.2, random_state=42)

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_data, train_labels)).batch(batch_size, drop_remainder=True)
val_ds = tf.data.Dataset.from_tensor_slices((test_data, test_labels)).batch(batch_size, drop_remainder=True)

# Prepare for saving model checkpoints
timestamp = datetime.now().strftime('%Y-%m-%d-%H:%M:%S')
checkpoint_dir = f"TPCNet/O16_dw_models/{timestamp}/weights"
checkpoint_path = f"{checkpoint_dir}/cp-{{epoch:03d}}.ckpt"
os.makedirs(checkpoint_dir, exist_ok=True)

# Callbacks for training
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, save_weights_only=True, monitor='val_accuracy', mode='max', save_best_only=False, save_freq='epoch')
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, mode='auto', verbose=1, min_delta=0.001, min_lr=0.00001)

# Compile and fit model
model.summary()
model.compile(loss="sparse_categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), metrics=["sparse_categorical_accuracy", "accuracy"])
history = model.fit(train_ds, validation_data=val_ds, epochs=num_epochs, callbacks=[checkpoint_callback, reduce_lr], verbose=1)

# Optional: Save learning curve plot
plot_learning_curve(history, 'TPCNet/O16_dw_models/plots/learning_curve.png')
